# Stage 2 实验 — Step 2：模型框架与训练循环

**训练数据（与 Stage 1 完全不同）**：
- 来源：`train_data/{lang}/articles/` + `labels-spans.txt`（CheckThat! 2024 原始字符级标注）
- 翻译文章（`translation_*/`）无字符级标注，Stage 2 **不使用**
- 规模：EN=518篇/9002spans, PL=172篇/61184spans, RU=190篇/4138spans


**评估集**：`overlap_analysis_results/{lang}_overlap_articles/`（SemEval dev ∩ CheckThat 2024 字符级金标准）

In [ ]:
# ============================================================
# CONFIGURATION — update these paths before running
# ============================================================
import os

# Root directory containing data/, results/, technique_list/, etc.
BASE = "your/path/here"  # e.g. "/home/user/propaganda-span-detection"

# SemEval-2023 / CheckThat!-2024 overlap evaluation set
# Expected subdirectories: en_overlap_articles/, po_overlap_articles/, ru_overlap_articles/
OVERLAP_DIR = "your/path/here"  # typically BASE + "/data_analy/overlap_analysis_results"

# CheckThat! 2024 Task 3 training data bundle
CHECKTHAT24_DIR = "your/path/here"  # root of the official CheckThat!-2024 data bundle

# S3 credentials (or set as environment variables)
os.environ.setdefault("AWS_ACCESS_KEY_ID",     "your-access-key-id")
os.environ.setdefault("AWS_SECRET_ACCESS_KEY", "your-secret-access-key")
S3_ENDPOINT = "https://your-s3-endpoint"
S3_BUCKET   = "your-s3-bucket"


In [ ]:
# ============================================================
# 0. 路径配置
# ============================================================
import os

NOTEBOOKS_DIR  = os.path.join(
    f"your/path/here",
    'Total_work', 'research_projects', 'disinformation_detection', 'notebooks'
)
OVERLAP_DIR    = os.path.join(NOTEBOOKS_DIR, 'data_analy', 'overlap_analysis_results')
TRAIN_DATA_DIR = os.path.join(NOTEBOOKS_DIR, 'data_analy', 'train_data')


# 训练集（Stage 2 数据准备脚本输出）
TRAIN_DATA = {
    'EN': {'articles': os.path.join(TRAIN_DATA_DIR, 'en', 'articles'),
           'spans'   : os.path.join(TRAIN_DATA_DIR, 'en', 'labels-spans.txt')},
    'PL': {'articles': os.path.join(TRAIN_DATA_DIR, 'pl', 'articles'),
           'spans'   : os.path.join(TRAIN_DATA_DIR, 'pl', 'labels-spans.txt')},
    'RU': {'articles': os.path.join(TRAIN_DATA_DIR, 'ru', 'articles'),
           'spans'   : os.path.join(TRAIN_DATA_DIR, 'ru', 'labels-spans.txt')},
}

# 评估集（SemEval dev ∩ CheckThat 2024）
EVAL_DATA = {
    'EN': {'articles': os.path.join(OVERLAP_DIR, 'en_overlap_articles', 'articles'),
           'spans'   : os.path.join(OVERLAP_DIR, 'en_overlap_articles', 'labels-subtask-3-spans.txt')},
    'PL': {'articles': os.path.join(OVERLAP_DIR, 'po_overlap_articles', 'articles'),
           'spans'   : os.path.join(OVERLAP_DIR, 'po_overlap_articles', 'labels-subtask-3-spans.txt')},
    'RU': {'articles': os.path.join(OVERLAP_DIR, 'ru_overlap_articles', 'articles'),
           'spans'   : os.path.join(OVERLAP_DIR, 'ru_overlap_articles', 'labels-subtask-3-spans.txt')},
}

print('=== 路径验证 ===')
for split, data_dict in [('TRAIN', TRAIN_DATA), ('EVAL', EVAL_DATA)]:
    for lang, paths in data_dict.items():
        for k, p in paths.items():
            print(f'  {split} {lang} {k}: {"✅" if os.path.exists(p) else "❌"} {os.path.basename(p)}')

In [ ]:
# ============================================================
# 0.0. 配置S3
# ============================================================

import boto3, io
from botocore.client import Config

ENDPOINT              = "https://your-s3-endpoint"  # Set to your S3-compatible endpoint
S3_BUCKET             = "your-s3-bucket"
S3_PREFIX             = 'multi_label_models'
AWS_ACCESS_KEY_ID = os.environ.get("AWS_ACCESS_KEY_ID", "your-access-key-id")
AWS_SECRET_ACCESS_KEY = os.environ.get("AWS_SECRET_ACCESS_KEY", "your-secret-access-key")

s3 = boto3.client(
    's3', endpoint_url=ENDPOINT,
    aws_access_key_id=AWS_ACCESS_KEY_ID,
    aws_secret_access_key=AWS_SECRET_ACCESS_KEY,
    config=Config(signature_version='s3v4'),
)

def s3_save_model(state_dict, s3_key):
    buf = io.BytesIO()
    torch.save(state_dict, buf)
    buf.seek(0)
    s3.upload_fileobj(buf, S3_BUCKET, s3_key)

def s3_load_model(s3_key, device):
    buf = io.BytesIO()
    s3.download_fileobj(S3_BUCKET, s3_key, buf)
    buf.seek(0)
    return torch.load(buf, map_location=device)

def s3_save_json(obj, s3_key):
    buf = io.BytesIO(json.dumps(obj, indent=2).encode('utf-8'))
    s3.upload_fileobj(buf, S3_BUCKET, s3_key)

print('✅ S3 工具Functions defined')

In [ ]:
# ============================================================
# 1. 依赖 + 设备
# ============================================================
import subprocess, sys, torch, numpy as np

for pkg in ['transformers', 'tqdm']:
    try:
        __import__(pkg)
    except ImportError:
        subprocess.run([sys.executable, '-m', 'pip', 'install', pkg], check=True)

print(f'torch  : {torch.__version__}')
print(f'CUDA   : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {DEVICE}')

In [ ]:
# ============================================================
# 2. 全局常量
# ============================================================
TECHNIQUES_FILE = "your/path/here"

with open(TECHNIQUES_FILE, 'r', encoding='utf-8') as f:
    TECHNIQUES = [line.strip() for line in f if line.strip()]

TECH2IDX   = {t: i for i, t in enumerate(TECHNIQUES)}
NUM_LABELS = 1 + len(TECHNIQUES) * 2
print(f'已加载 {len(TECHNIQUES)} 种技术，NUM_LABELS = {NUM_LABELS}  (0=O, B/I x{len(TECHNIQUES)})')
print(f'技术列表: {TECHNIQUES}')

In [ ]:
# ============================================================
# 3. 数据解析
#    parse_span_line: 兼容 EN 5列 / PL·RU 4列
#    validate_spans : 自动跳过越界 span
# ============================================================

def parse_span_line(parts):
    """
    5列 (EN): row_num  article_id  technique  start  end
    4列 (PL/RU):       article_id  technique  start  end
    返回 (article_id, technique, start, end) 或 None
    """
    if len(parts) == 5:
        _, art, tech, s, e = parts
    elif len(parts) == 4:
        art, tech, s, e = parts
    else:
        return None
    try:
        return art.strip(), tech.strip(), int(s), int(e)
    except ValueError:
        return None


def parse_spans_file(path, article_dir=None):
    """
    解析标注文件 → {article_id: [(start, end, technique), ...]}
    article_dir 非 None 时启用越界校验（跳过非法 span）
    """
    # 预加载文章长度缓存
    len_cache = {}
    def art_len(aid):
        if aid not in len_cache:
            for fname in [f'{aid}.txt', f'article{aid}.txt']:
                p = os.path.join(article_dir, fname)
                if os.path.exists(p):
                    with open(p, 'rb') as f:
                        len_cache[aid] = len(f.read().decode('utf-8', errors='replace'))
                    return len_cache[aid]
            len_cache[aid] = None
        return len_cache[aid]

    spans, skipped = {}, 0
    with open(path, 'r', encoding='utf-8', errors='replace') as f:
        for line in f:
            parsed = parse_span_line(line.rstrip('\n').split('\t'))
            if parsed is None:
                continue
            aid, tech, s, e = parsed
            if tech not in TECH2IDX:
                continue
            if article_dir is not None:
                al = art_len(aid)
                if al is not None and (s < 0 or e > al or s >= e):
                    skipped += 1; continue
            spans.setdefault(aid, []).append((s, e, tech))
    if skipped:
        print(f'  ⚠️  跳过 {skipped} 条越界 span')
    return spans


def read_article(art_dir, art_id):
    for fname in [f'{art_id}.txt', f'article{art_id}.txt']:
        p = os.path.join(art_dir, fname)
        if os.path.exists(p):
            with open(p, 'rb') as f:
                return f.read().decode('utf-8', errors='replace')
    return None


def load_dataset(data_config, languages, validate_spans=True):
    """
    返回 {lang: [(art_id, text, [(s,e,tech),...]), ...]}
    """
    result = {}
    for lang in languages:
        art_dir = data_config[lang]['articles']
        sp_path = data_config[lang]['spans']
        if not os.path.exists(sp_path) or not os.path.exists(art_dir):
            print(f'  ❌ {lang}: 路径不存在')
            result[lang] = []; continue
        spans_dict = parse_spans_file(sp_path, art_dir if validate_spans else None)
        samples, miss = [], 0
        for aid, sps in spans_dict.items():
            text = read_article(art_dir, aid)
            if text is None:
                miss += 1; continue
            samples.append((aid, text, sps))
        n_sp = sum(len(s[2]) for s in samples)
        print(f'  {lang}: {len(samples)} 篇, {n_sp} spans'
              + (f' (⚠️ {miss} 篇文章缺失)' if miss else ''))
        result[lang] = samples
    return result


print('数据工具Functions defined ✅')

In [ ]:
# ============================================================
# 4. BIO 转换
# ============================================================

def convert_to_bio(text, char_spans, tokenizer, max_length=256):
    enc = tokenizer(
        text, max_length=max_length, padding='max_length',
        truncation=True, return_offsets_mapping=True, return_token_type_ids=True
    )
    labels = [0] * max_length
    for s, e, tech in char_spans:
        if tech not in TECH2IDX: continue
        b, i_tag, first = 1 + TECH2IDX[tech]*2, 2 + TECH2IDX[tech]*2, True
        for idx, (ts, te) in enumerate(enc['offset_mapping']):
            if ts == te: continue
            if ts < e and te > s:
                labels[idx] = b if first else i_tag; first = False
    return enc, labels


print('BIO 转换Functions defined ✅')

In [ ]:
# ============================================================
# 5. Dataset
# ============================================================
import torch
from torch.utils.data import Dataset, DataLoader

class PropagandaSpanDataset(Dataset):
    def __init__(self, samples, tokenizer, max_length=256, stride=128):
        self.items = []
        cw, cs = max_length * 4, stride * 4   # char window / stride
        for _, text, spans in samples:
            tl, start = len(text), 0
            while start < tl:
                end = min(start + cw, tl)
                seg = text[start:end]
                local = [(max(s-start,0), min(e-start,end-start), t)
                         for s,e,t in spans if s < end and e > start]
                enc, labs = convert_to_bio(seg, local, tokenizer, max_length)
                self.items.append((
                    torch.tensor(enc['input_ids'],      dtype=torch.long),
                    torch.tensor(enc['attention_mask'], dtype=torch.long),
                    torch.tensor(enc.get('token_type_ids',[0]*max_length), dtype=torch.long),
                    torch.tensor(labs,                  dtype=torch.long),
                ))
                if end == tl: break
                start += cs
        print(f'Dataset: {len(self.items)} windows from {len(samples)} articles')

    def __len__(self): return len(self.items)
    def __getitem__(self, idx):
        ii, am, tt, lb = self.items[idx]
        return {'input_ids': ii, 'attention_mask': am,
                'token_type_ids': tt, 'labels': lb}


print('Dataset 定义完成 ✅')

In [ ]:
# ============================================================
# 6. TokenClassifier（mBERT / XLM-R，5种特征模式）
# ============================================================
import torch.nn as nn
from transformers import AutoModel

class TokenClassifier(nn.Module):
    """
    Config-A: single_layer, layer=12  (默认最后层)
    Config-B: single_layer, layer=10  (Mela 提交)
    Config-C: single_layer, layer=8   (Mela 消融最优)
    Config-D: avg_pool
    Config-E: learned_weight
    """
    def __init__(self, model_name, num_labels=NUM_LABELS,
                 feature_mode='single_layer', target_layer=12, dropout=0.1):
        super().__init__()
        self.encoder      = AutoModel.from_pretrained(model_name, output_hidden_states=True)
        self.feature_mode = feature_mode
        self.target_layer = target_layer
        hidden_size       = self.encoder.config.hidden_size
        if feature_mode == 'learned_weight':
            self.layer_weights = nn.Parameter(torch.ones(12))
        self.dropout    = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden_size, num_labels)

    def forward(self, input_ids, attention_mask, token_type_ids=None):
        kw = dict(input_ids=input_ids, attention_mask=attention_mask,
                  output_hidden_states=True)
        if (token_type_ids is not None
                and getattr(self.encoder.config, 'type_vocab_size', 1) > 1):
            kw['token_type_ids'] = token_type_ids
        hs = self.encoder(**kw).hidden_states   # tuple len=13

        if self.feature_mode == 'single_layer':
            h = hs[self.target_layer]
        elif self.feature_mode == 'avg_pool':
            h = torch.stack(hs[1:], 0).mean(0)
        elif self.feature_mode == 'learned_weight':
            w = torch.softmax(self.layer_weights, 0)
            h = (torch.stack(hs[1:], 0) * w.view(12,1,1,1)).sum(0)
        else:
            raise ValueError(self.feature_mode)

        return self.classifier(self.dropout(h))

    def get_layer_weights(self):
        assert self.feature_mode == 'learned_weight'
        return torch.softmax(self.layer_weights, 0).detach().cpu().numpy()


print('TokenClassifier 定义完成 ✅')

In [ ]:
# ============================================================
# 7. 损失函数
# ============================================================

def compute_loss(logits, labels, attention_mask, num_labels=NUM_LABELS):
    w = torch.ones(num_labels, device=logits.device)
    w[0] = 0.5; w[1:] = 2.0
    active = labels.clone()
    active[attention_mask == 0] = -100
    return nn.CrossEntropyLoss(weight=w, ignore_index=-100)(
        logits.view(-1, num_labels), active.view(-1)
    )

print('损失Functions defined ✅')

In [ ]:
# ============================================================
# 8. Span F1 评估
# ============================================================

def decode_spans(logits, offset_map, win_start):
    preds = logits.argmax(-1).tolist()
    spans, i = [], 0
    while i < len(preds):
        p = preds[i]
        if p == 0 or p % 2 == 0: i += 1; continue
        ts, te = offset_map[i]
        s, e = win_start + ts, win_start + te
        tech = TECHNIQUES[(p-1)//2]
        j = i + 1
        while j < len(preds) and preds[j] == p+1:
            e = win_start + offset_map[j][1]; j += 1
        spans.append((s, e, tech)); i = j
    return spans


def overlap_f1(pred, gold):
    pool = list(gold); tp = fp = 0
    for ps, pe, pt in pred:
        hit = False
        for i, (gs, ge, gt) in enumerate(pool):
            if gt == pt and ps < ge and pe > gs:
                tp += 1; pool.pop(i); hit = True; break
        if not hit: fp += 1
    return tp, fp, len(pool)


def evaluate_model(model, samples, tokenizer, device, max_length=256, stride=128):
    model.eval()
    cw, cs = max_length*4, stride*4
    TP = FP = FN = 0
    with torch.no_grad():
        for _, text, gold in samples:
            tl, ws, preds = len(text), 0, []
            while ws < tl:
                we = min(ws + cw, tl)
                enc = tokenizer(text[ws:we], max_length=max_length,
                                padding='max_length', truncation=True,
                                return_offsets_mapping=True, return_tensors='pt')
                om = enc.pop('offset_mapping')[0].tolist()
                tt = enc.get('token_type_ids', torch.zeros_like(enc['input_ids']))
                logits = model(enc['input_ids'].to(device),
                               enc['attention_mask'].to(device),
                               tt.to(device))[0].cpu()
                preds += decode_spans(logits, om, ws)
                if we == tl: break
                ws += cs
            tp, fp, fn = overlap_f1(preds, gold)
            TP += tp; FP += fp; FN += fn
    P = TP/(TP+FP) if TP+FP else 0.0
    R = TP/(TP+FN) if TP+FN else 0.0
    F = 2*P*R/(P+R) if P+R else 0.0
    return {'f1':F, 'precision':P, 'recall':R, 'tp':TP, 'fp':FP, 'fn':FN}


print('评估Functions defined ✅')

In [ ]:
# ============================================================
# 9. 训练循环
# ============================================================
from torch.optim import AdamW
from torch.optim.lr_scheduler import LambdaLR
from tqdm.auto import tqdm
import json


def train_model(config, train_samples, eval_dict, device):
    from transformers import AutoTokenizer
    torch.manual_seed(config.get('seed', 42))
    np.random.seed(config.get('seed', 42))

    tokenizer  = AutoTokenizer.from_pretrained(config['model_name'])
    max_length = config.get('max_length', 256)
    stride     = config.get('stride', 128)

    print('构建 Dataset...')
    ds = PropagandaSpanDataset(train_samples, tokenizer, max_length, stride)
    dl = DataLoader(ds, batch_size=config['batch_size'],
                    shuffle=True, num_workers=0) # num_workers=4)

    print(f'初始化模型: {config["model_name"]} | '
          f'{config["feature_mode"]} | layer={config.get("target_layer","N/A")}')
    model = TokenClassifier(
        config['model_name'], NUM_LABELS,
        config['feature_mode'], config.get('target_layer', 12)
    ).to(device)

    optimizer    = AdamW(model.parameters(), lr=config['learning_rate'],
                         weight_decay=config.get('weight_decay', 0.01))
    total_steps  = len(dl) * config['num_epochs']
    warmup_steps = config.get('warmup_steps', 0)

    def lr_fn(step):
        if step < warmup_steps:
            return step / max(1, warmup_steps)
        return max(0.0, (total_steps - step) / max(1, total_steps - warmup_steps))
    scheduler = LambdaLR(optimizer, lr_fn) if warmup_steps > 0 else None

    use_fp16 = config.get('fp16', False) and device.type == 'cuda'
    scaler   = torch.cuda.amp.GradScaler() if use_fp16 else None

    # out_dir   = config.get('output_dir', OUTPUT_DIR)
    # os.makedirs(out_dir, exist_ok=True)
    # best_ckpt = os.path.join(out_dir, 'best_model.pt')

    run_name = config.get('run_name', 'run')
    s3_model_key   = f'{S3_PREFIX}/{run_name}/best_model.pt'
    s3_history_key = f'{S3_PREFIX}/{run_name}/history.json'
    print(f'S3 路径: s3://{S3_BUCKET}/{s3_model_key}')
    history   = {'train_loss': [], 'eval': {}}
    best_f1   = -1.0

    for epoch in range(1, config['num_epochs'] + 1):
        model.train()
        epoch_loss = 0.0
        for batch in tqdm(dl, desc=f'Epoch {epoch}/{config["num_epochs"]}'):
            ids  = batch['input_ids'].to(device)
            mask = batch['attention_mask'].to(device)
            tt   = batch['token_type_ids'].to(device)
            labs = batch['labels'].to(device)
            optimizer.zero_grad()
            if use_fp16:
                with torch.cuda.amp.autocast():
                    loss = compute_loss(model(ids, mask, tt), labs, mask)
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer); scaler.update()
            else:
                loss = compute_loss(model(ids, mask, tt), labs, mask)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
            if scheduler: scheduler.step()
            epoch_loss += loss.item()

        avg = epoch_loss / len(dl)
        history['train_loss'].append(avg)
        print(f'Epoch {epoch}: loss={avg:.4f}')

        epoch_eval = {}
        for lang, samps in eval_dict.items():
            m = evaluate_model(model, samps, tokenizer, device, max_length, stride)
            epoch_eval[lang] = m
            print(f'  {lang}: F1={m["f1"]*100:.2f}%  '
                  f'P={m["precision"]*100:.2f}%  R={m["recall"]*100:.2f}%  '
                  f'TP={m["tp"]} FP={m["fp"]} FN={m["fn"]}')
        history['eval'][epoch] = epoch_eval

        cur_f1 = epoch_eval[list(epoch_eval)[0]]['f1']
        if cur_f1 > best_f1:
            best_f1 = cur_f1
            # torch.save(model.state_dict(), best_ckpt)
            # print(f'  💾 Best checkpoint saved (F1={best_f1*100:.2f}%)')
            s3_save_model(model.state_dict(), s3_model_key)
            print(f'  💾 Best checkpoint → S3 (F1={best_f1*100:.2f}%)')

    # with open(os.path.join(out_dir, 'history.json'), 'w') as f:
    #     json.dump(history, f, indent=2)
    s3_save_json(history, s3_history_key)
    print(f'\n训练完成！Best F1={best_f1*100:.2f}%')
    # model.load_state_dict(torch.load(best_ckpt, map_location=device))
    model.load_state_dict(s3_load_model(s3_model_key, device))
    return model, history


print('训练循环定义完成 ✅')

In [ ]:
# ============================================================
# 10. Sanity Check（不需要真实数据） 
# 正式实验时不需要运行这个CELL
# ============================================================
from transformers import AutoTokenizer
print('=== Sanity Check ===')

tok = AutoTokenizer.from_pretrained('bert-base-multilingual-cased')
dummy_text  = 'Propaganda test sentence. ' * 20
dummy_spans = [(10, 50, 'Loaded_Language'), (100, 130, 'Doubt')]
enc, labs   = convert_to_bio(dummy_text, dummy_spans, tok, 64)

ids  = torch.tensor([enc['input_ids']])
mask = torch.tensor([enc['attention_mask']])
tt   = torch.tensor([enc.get('token_type_ids', [0]*64)])
lab  = torch.tensor([labs])

for name, mode, layer in [('Config-C', 'single_layer', 8),
                            ('Config-E', 'learned_weight', None)]:
    kw = dict(feature_mode=mode)
    if layer: kw['target_layer'] = layer
    m = TokenClassifier('bert-base-multilingual-cased', **kw)
    with torch.no_grad():
        out = m(ids, mask, tt)
    loss = compute_loss(out, lab, mask)
    print(f'  {name}: logits={out.shape}  loss={loss.item():.4f}')
    if mode == 'learned_weight':
        print(f'         weights={m.get_layer_weights().round(3)}')

print('Sanity Check 通过 ✅')

In [ ]:
# ============================================================
# Cell 11：多语言数据加载（EN + PL + RU）
# ============================================================
from transformers import AutoTokenizer

print('=== 加载训练集 ===')
train_data = load_dataset(TRAIN_DATA, ['EN', 'PL', 'RU'])

print('\n=== 加载评估集 ===')
eval_by_lang = load_dataset(EVAL_DATA, ['EN', 'PL', 'RU'])

# 合并三语言训练样本
train_samples = []
for lang, samples in train_data.items():
    train_samples.extend(samples)

print(f'\n[TRAIN] 合并后总计: {len(train_samples)} 篇')
for lang, samps in eval_by_lang.items():
    print(f'[EVAL]  {lang}: {len(samps)} 篇')

---
## Cell 12-S2C：训练配置（S2-C：mBERT + L10，多语言）

消融结论：mBERT 最优层 = L10（Config-B，F1=25.82%）  
本实验：在 EN+PL+RU 合并数据上训练，生成多语言 S2-C 模型

In [ ]:
# # ============================================================
# # Cell 12（S2-C）：mBERT + L10 多语言训练配置
# # ============================================================
# #
# # 实验一：Stage 2 方法主对比实验
# # 固定 Stage 1 = Sup-FT，对比 4 种 Stage 2 方法
# #
# # ┌────────┬──────────────────┬────────────┬──────────┬────────────────────────────────────────────┐
# # │ 方法    │ 骨干             │ 特征层     │ 是否训练 │ run_name                                    │
# # ├────────┼──────────────────┼────────────┼──────────┼────────────────────────────────────────────┤
# # │ S2-A   │ GPT-4o-mini      │ —          │ ❌       │ （已有结果，见 baseline TSV）               │
# # │ S2-B   │ GPT-4o-mini      │ —          │ ❌       │ （已有结果，见 baseline TSV）               │
# # │ S2-C ✳ │ mBERT            │ L10        │ ✅       │ exp1_s2c_mbert_l10_multilingual            │
# # │ S2-D   │ XLM-RoBERTa      │ L10        │ ✅       │ exp1_s2d_xlmr_l10_multilingual             │
# # └────────┴──────────────────┴────────────┴──────────┴────────────────────────────────────────────┘
# #
# # 层选择依据：
# #   - 消融 2a (mBERT)   : Config-B (L10) F1=25.82% 最优，且 Config-E 权重峰值在 L9-L10
# #   - Config-A (L12) Best Epoch=5，在多语言场景下 early-stop 风险更大，放弃
# #   - 两模型统一用 L10，论文叙述更清晰
# #
# # 当前运行配置：S2-C（mBERT + L10）
# # ============================================================
# CONFIG = {
#     'model_name'   : 'bert-base-multilingual-cased',   # mBERT
#     'feature_mode' : 'single_layer',
#     'target_layer' : 10,                               # 消融 2a 最优层

#     'num_epochs'   : 30,        
#     'batch_size'   : 32,
#     'learning_rate': 6.91e-5,   # Mela Optuna 最优值（mBERT 专用）
#     'weight_decay' : 0.00358,
#     'warmup_steps' : 226,

#     'fp16'         : True,
#     'seed'         : 42,
#     'max_length'   : 256,
#     'stride'       : 128,

#     # 多语言：训练集合并 EN+PL+RU，评估集分语言独立评估
#     'languages'    : ['en', 'pl', 'ru'],

#     'run_name'     : 'exp1_s2c_mbert_l10_multilingual',
# }
# print('=== CONFIG (S2-C: mBERT L10 Multilingual) ===')
# for k, v in CONFIG.items():
#     print(f'  {k:20s}: {v}')

---
## Cell 12-S2D：训练配置（S2-D：XLM-RoBERTa + L10，多语言）

消融结论：
- XLM-R Config-A（L12）Best Epoch=5，FP=1922，在多语言场景下 early-stopping 风险较大  
- Config-E 权重峰值在 L10（0.0961），与 mBERT 一致  
- 统一选 L10，论文叙述：「消融实验表明 L10 是两种骨干的最优层」

In [ ]:
# ============================================================
# Cell 12（S2-D）：XLM-RoBERTa + L10 多语言训练配置
# ============================================================
#
# 当前运行配置：S2-D（XLM-RoBERTa + L10）
# 跑完 S2-C 后，将下方 CONFIG 替换到 Cell 12，重新运行 Cell 13
# ============================================================
CONFIG = {
    'model_name'   : 'xlm-roberta-base',               # XLM-RoBERTa
    'feature_mode' : 'single_layer',
    'target_layer' : 10,                               # 消融 2b 权重分析峰值层
                                                       # （放弃 L12：Best Epoch=5，多语言 early-stop 风险大）

    'num_epochs'   : 30,        # 消融 2b 显示 30 epoch 足够收敛
    'batch_size'   : 16,        # XLM-R 显存占用更大，batch 减半
    'learning_rate': 2e-5,  # 原本是5e-5但是不稳定
    'weight_decay' : 0.01,
     'warmup_steps' : 200,         # 0的时候Loss 持续下降到接近 0（严重过拟合），但 Span F1 在 epoch 4 之后就开始剧烈震荡并持续下滑

    'fp16'         : True,
    'seed'         : 42,
    'max_length'   : 256,
    'stride'       : 128,

    # 多语言：训练集合并 EN+PL+RU，评估集分语言独立评估
    'languages'    : ['en', 'pl', 'ru'],

    'run_name'     : 'exp1_s2d_xlmr_l10_warmup_step200_lr2e5_multilingual',
}
print('=== CONFIG (S2-D: XLM-RoBERTa L10 Multilingual) ===')
for k, v in CONFIG.items():
    print(f'  {k:20s}: {v}')

In [ ]:
# 读取评估集文章ID
eval_ids = set()
with open(EVAL_DATA['EN']['spans']) as f:
    for line in f:
        parts = line.strip().split('\t')
        if parts and parts[0]:
            eval_ids.add(parts[0].replace('article', ''))

# 读取训练集文章ID
train_ids = set()
with open(TRAIN_DATA['EN']['spans']) as f:
    for line in f:
        parts = line.strip().split('\t')
        if parts and parts[0]:
            train_ids.add(parts[0].replace('article', ''))

overlap = eval_ids & train_ids
print(f'评估集中有 {len(overlap)}/{len(eval_ids)} 篇文章在训练集里')
print(f'重叠文章ID示例: {list(overlap)[:5]}')

In [ ]:
# Cell 13（无需修改）
if train_samples and eval_by_lang:
    model, history = train_model(CONFIG, train_samples, eval_by_lang, DEVICE)
else:
    print('⚠️  请先运行 Cell 11 加载数据')

In [ ]:
# ============================================================
# 14. 训练曲线
# ============================================================
import matplotlib.pyplot as plt

if 'history' in dir() and history['train_loss']:
    epochs = sorted(history['eval'])
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(range(1, len(history['train_loss'])+1), history['train_loss'], 'o-')
    axes[0].set(xlabel='Epoch', ylabel='Loss', title='Train Loss'); axes[0].grid(True)
    for lang in next(iter(history['eval'].values())):
        axes[1].plot(epochs, [history['eval'][e][lang]['f1']*100 for e in epochs],
                     'o-', label=lang)
    axes[1].set(xlabel='Epoch', ylabel='Span F1 (%)', title='Span F1')
    axes[1].legend(); axes[1].grid(True)
    plt.suptitle(f"{CONFIG['model_name']} | {CONFIG['feature_mode']} L{CONFIG.get('target_layer','')}")
    plt.tight_layout()
    # plt.savefig(os.path.join(CONFIG['output_dir'], 'training_curve.png'), dpi=150)
    buf = io.BytesIO()
    plt.savefig(buf, format='png', dpi=150)
    buf.seek(0)
    s3.upload_fileobj(buf, S3_BUCKET, f"{S3_PREFIX}/{CONFIG['run_name']}/training_curve.png")
    plt.show()

In [ ]:
# ============================================================
# 15. Config-E 层权重可视化
# ============================================================
import matplotlib.pyplot as plt, numpy as np

def visualize_layer_weights(model, label, save_dir=None):
    w = model.get_layer_weights()
    layers = [f'L{i+1}' for i in range(12)]
    mean_w = 1/12
    fig, ax = plt.subplots(figsize=(10,4))
    ax.bar(layers, w, color=['#1565C0' if x > mean_w else '#90CAF9' for x in w])
    ax.axhline(mean_w, color='red', linestyle='--', alpha=0.7, label='Uniform')
    mi = int(np.argmax(w))
    ax.annotate(f'Max: L{mi+1} ({w[mi]:.3f})',
                xy=(mi, w[mi]), xytext=(min(mi+1.5,10), w[mi]+0.005),
                arrowprops=dict(arrowstyle='->', color='darkred'))
    ax.set(xlabel='Layer', ylabel='Weight', title=f'Config-E — {label}')
    ax.legend(); plt.tight_layout()
    if save_dir: plt.savefig(os.path.join(save_dir,'layer_weights.png'), dpi=150)
    plt.show()
    for l, wi in zip(layers, w):
        print(f'  {l}: {"█"*max(1,int(wi*200)):<25} {wi:.4f}{" ← max" if wi==w.max() else ""}')

if 'model' in dir() and CONFIG.get('feature_mode') == 'learned_weight':
    visualize_layer_weights(model, CONFIG['model_name'], save_dir=None)
    # 保存到S3
    buf = io.BytesIO()
    plt.savefig(buf, format='png', dpi=150)
    buf.seek(0)
    s3.upload_fileobj(buf, S3_BUCKET, 
                      f"{S3_PREFIX}/{CONFIG['run_name']}/layer_weights.png")
    print(f"✅ 层权重图 → S3: {S3_PREFIX}/{CONFIG['run_name']}/layer_weights.png")
else:
    print('（feature_mode=learned_weight 时才运行）')

In [ ]:
# ============================================================
# Cell 16b：三语言循环推理
# ============================================================
import io
import torch
import pandas as pd
from transformers import AutoTokenizer

# ------------------------------------------------------------------
# 0. 准备
# ------------------------------------------------------------------
model.eval()
tokenizer_inf = AutoTokenizer.from_pretrained(CONFIG['model_name'])
max_length = CONFIG['max_length']
stride     = CONFIG['stride']
run_name   = CONFIG['run_name']

# 字符窗口宽度（token -> char 的粗略换算，与原 Cell 16 保持一致）
CHAR_WINDOW = max_length * 4
CHAR_STRIDE = stride * 4

# ------------------------------------------------------------------
# 1. decode_spans（与原 Cell 16 完全相同，此处内联保证独立可运行）
# ------------------------------------------------------------------
def decode_spans_from_logits(logits, offset_mapping, char_offset=0):
    """
    将单窗口 logits 解码为 (abs_char_start, abs_char_end, technique, confidence) 列表
    logits: (seq_len, num_labels) tensor，已在 CPU
    offset_mapping: list of (tok_start, tok_end)，相对于当前窗口
    char_offset: 当前窗口在文章中的起始字符位置
    """
    probs = torch.softmax(logits, dim=-1)          # (seq_len, 47)
    preds = probs.argmax(dim=-1).tolist()           # (seq_len,)

    spans = []
    i = 0
    while i < len(preds):
        label = preds[i]
        if label == 0 or offset_mapping[i][0] == offset_mapping[i][1]:
            i += 1
            continue
        if label % 2 == 1:   # B- 标签
            tech_idx  = (label - 1) // 2
            technique = TECHNIQUES[tech_idx]
            span_start = offset_mapping[i][0] + char_offset
            span_end   = offset_mapping[i][1] + char_offset
            conf       = probs[i][label].item()
            j = i + 1
            while j < len(preds) and preds[j] == label + 1:   # I- 标签
                if offset_mapping[j][0] != offset_mapping[j][1]:
                    span_end = offset_mapping[j][1] + char_offset
                j += 1
            spans.append((span_start, span_end, technique, round(conf, 4)))
            i = j
        else:
            i += 1
    return spans


# ------------------------------------------------------------------
# 2. 单语言推理函数
# ------------------------------------------------------------------
def infer_one_language(lang_key, samples):
    """
    samples: load_dataset() 返回的 [(art_id, text, spans), ...] tuple 列表
    """
    rows = []
    with torch.no_grad():
        for art_id, text, _ in samples:   # tuple 解包，spans 推理时不需要
            text_len = len(text)
            ws = 0
            while ws < text_len:
                we    = min(ws + CHAR_WINDOW, text_len)
                chunk = text[ws:we]
                enc = tokenizer_inf(
                    chunk,
                    max_length=max_length,
                    padding='max_length',
                    truncation=True,
                    return_offsets_mapping=True,
                    return_tensors='pt'
                )
                offset_mapping = enc.pop('offset_mapping')[0].tolist()
                tt = enc.get('token_type_ids',
                             torch.zeros_like(enc['input_ids']))
                logits = model(
                    enc['input_ids'].to(DEVICE),
                    enc['attention_mask'].to(DEVICE),
                    tt.to(DEVICE)
                )[0].cpu()
                spans = decode_spans_from_logits(logits, offset_mapping, char_offset=ws)
                for s, e, tech, conf in spans:
                    rows.append({
                        'article_id': art_id,
                        'technique' : tech,
                        'start'     : s,
                        'end'       : e,
                        'confidence': conf,
                        'source'    : run_name,
                    })
                if we == text_len:
                    break
                ws += CHAR_STRIDE
    df = pd.DataFrame(rows, columns=['article_id','technique','start','end','confidence','source'])
    return df


# 三语言循环推理（key 大写，与 eval_by_lang 一致）
results_by_lang = {}
for lang in ('EN', 'PL', 'RU'):
    print(f'\n[{lang}] 开始推理 ({len(eval_by_lang[lang])} 篇)...')
    df_pred = infer_one_language(lang, eval_by_lang[lang])
    results_by_lang[lang] = df_pred

    s3_key = f'{S3_PREFIX}/{run_name}/predictions_{lang.lower()}.tsv'
    buf = io.BytesIO(df_pred.to_csv(sep='\t', index=False).encode('utf-8'))
    s3.upload_fileobj(buf, S3_BUCKET, s3_key)
    print(f'  预测 span 数 : {len(df_pred)}')
    print(f'  ✅ 已存 S3  : {s3_key}')

print('\n=== 三语言推理完成 ===')

In [ ]:
# ------------------------------------------------------------------
# 4. 快速统计（推理结果概览，方便确认输出是否合理）
# ------------------------------------------------------------------
print(f'运行名称: {run_name}\n')
print(f'{"语言":<6} {"span总数":>8} {"涉及文章数":>10} {"平均span/文章":>14}')
print('-' * 42)
for lang in ('EN', 'PL', 'RU'):
    df = results_by_lang[lang]
    n_arts  = df['article_id'].nunique() if len(df) > 0 else 0
    n_spans = len(df)
    avg     = round(n_spans / max(n_arts, 1), 1)
    print(f'{lang:<6} {n_spans:>8} {n_arts:>10} {avg:>14}')

In [ ]:
# ============================================================
# 18. 单个Config训练结果总结
# ============================================================

def summarize_config(config, history):
    run_name = config.get('run_name', 'run')
    s3_base  = f's3://{S3_BUCKET}/{S3_PREFIX}/{run_name}'

    # 找best epoch
    best_f1, best_epoch, best_metrics = -1, -1, {}
    for epoch, langs in history['eval'].items():
        f1 = langs['EN']['f1']
        if f1 > best_f1:
            best_f1 = f1
            best_epoch = int(epoch)
            best_metrics = langs['EN']

    print('='*60)
    print('📋 训练配置')
    print('='*60)
    print(f'  模型        : {config["model_name"]}')
    print(f'  feature_mode: {config["feature_mode"]}')
    print(f'  target_layer: {config.get("target_layer", "—")}')
    print(f'  num_epochs  : {config["num_epochs"]}')
    print(f'  batch_size  : {config["batch_size"]}')
    print(f'  learning_rate: {config["learning_rate"]}')
    print(f'  weight_decay: {config["weight_decay"]}')
    print(f'  warmup_steps: {config["warmup_steps"]}')
    print(f'  fp16        : {config["fp16"]}')
    print(f'  seed        : {config["seed"]}')
    print(f'  max_length  : {config["max_length"]}')
    print(f'  stride      : {config["stride"]}')

    print('\n' + '='*60)
    print('📊 最优结果')
    print('='*60)
    print(f'  Best F1     : {best_f1*100:.2f}%')
    print(f'  Best Epoch  : {best_epoch}')
    print(f'  Precision   : {best_metrics["precision"]*100:.2f}%')
    print(f'  Recall      : {best_metrics["recall"]*100:.2f}%')
    print(f'  TP          : {best_metrics["tp"]}')
    print(f'  FP          : {best_metrics["fp"]}')
    print(f'  FN          : {best_metrics["fn"]}')

    print('\n' + '='*60)
    print('💾 S3 文件位置')
    print('='*60)
    print(f'  模型checkpoint : {s3_base}/best_model.pt')
    print(f'  训练历史       : {s3_base}/history.json')
    print(f'  Loss曲线图     : {s3_base}/training_curve.png')
    print(f'  预测结果(EN)   : {s3_base}/predictions_en.tsv')
    print(f'  预测结果(PL)   : {s3_base}/predictions_pl.tsv')
    print(f'  预测结果(RU)   : {s3_base}/predictions_ru.tsv')
    print('='*60)

summarize_config(CONFIG, history)

In [ ]:
import os

BASE = "your/path/here"

files = {
    'Sup-FT EN'   : BASE + 'model_tests/semeval_task3_en_dev_predictions_official_format.txt',
    'Sup-FT PL'   : BASE + 'model_tests/semeval_task3_po_dev_predictions_official_format.txt',
    'Sup-FT RU'   : BASE + 'model_tests/semeval_task3_ru_dev_predictions_official_format.txt',
    'Prompt-A EN' : BASE + 'LLM_tests/results_en_results_semeval_task3_dev_en_context_all.tsv',
    'Prompt-A PL' : BASE + 'LLM_tests/semeval2023_task3_dev_results_po_po_context_all.tsv',
    'Prompt-A RU' : BASE + 'LLM_tests/semeval2023_task3_dev_results_ru_ru_context_all.tsv',
    'Iter-Ens EN' : BASE + 'LLM_tests/results_en_results_semeval_task3_dev_en_voting_aggressive_all.tsv',
    'Iter-Ens PL' : BASE + 'LLM_tests/semeval2023_task3_dev_results_po_po_voting_aggressive_all.tsv',
    'Iter-Ens RU' : BASE + 'LLM_tests/semeval2023_task3_dev_results_ru_ru_voting_aggressive_all.tsv',
}

for name, path in files.items():
    status = '✅' if os.path.exists(path) else '❌'
    print(f'{status} {name}: {os.path.basename(path)}')

In [ ]:
with open("your/path/here", 'r') as f:
    print(f.read())